In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim

from tqdm import trange, tqdm
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
df = pd.read_csv(url, header=None)

In [45]:
label_map = {
    "Iris-setosa": 0,
    "Iris-versicolor": 1,
    "Iris-virginica": 2
}

In [46]:
GPU_indx = 0
device = torch.device(GPU_indx if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [47]:
train = df.sample(120).sort_index()
test = df.drop(train.index).sort_index()
train, test = train.reset_index(drop=True), test.reset_index(drop=True)

x_train, x_test = train.iloc[:, :4], test.iloc[:, :4]
y_train, y_test = train.iloc[:, -1].map(label_map), test.iloc[:, -1].map(label_map)

x_train, y_train, x_test, y_test = torch.tensor(x_train.to_numpy(), dtype=torch.float32).to(device), torch.tensor(y_train).to(device), torch.tensor(x_test.to_numpy(), dtype=torch.float32).to(device), torch.tensor(y_test).to(device)


In [48]:
class IrisDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [49]:
train_loader = DataLoader(IrisDataset(x_train, y_train), batch_size=9, shuffle=True)
test_loader = DataLoader(IrisDataset(x_test, y_test), batch_size=8, shuffle=False)

In [50]:
class NN(nn.Module):
    def __init__(self, num_classes):
        super(NN, self).__init__()

        self.fc1 = nn.Linear(4, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, num_classes)

    def forward(self, x):

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x 

In [51]:
model = NN(3).to(device)
criterion = nn.CrossEntropyLoss()
lr = 1e-3
optimizer = optim.Adam(model.parameters(), lr)
n_epochs = 20

print(model)

NN(
  (fc1): Linear(in_features=4, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=3, bias=True)
)


In [52]:
def train_epoch(model, train_loader, criterion, optimizer, loss_logger):
    for batch_idx, (data, target) in enumerate(tqdm(train_loader, desc="Training", leave=False)):
        
        outputs = model(data.to(device))
        loss = criterion(outputs, target.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        loss_logger.append(loss.item())

    return model, optimizer, loss_logger

In [53]:
def test_model(model, test_loader, criterion, loss_logger):
    with torch.no_grad():
        correct_predictions = 0
        total_predictions = 0
        
        for batch_idx, (data, target) in enumerate(tqdm(test_loader, desc="Testing", leave=False)):   
            
            outputs = model(data.to(device))        
            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == target.to(device)).sum().item()
            total_predictions += target.shape[0]
            loss = criterion(outputs, target.to(device))
            loss_logger.append(loss.item())
            
        acc = (correct_predictions/total_predictions) * 100.0
        return loss_logger, acc

In [54]:
# Create empty lists for the train/test losses and the test accuracy
train_loss = []
test_loss  = []
test_acc   = []

In [55]:
for i in trange(n_epochs, desc="Epoch", leave=False):
    # Call the training function to perform an epoch of training
    model, optimizer, train_loss = train_epoch(model, train_loader, criterion, optimizer, train_loss)
    
    # Call the testing function to work out the test loss and accuracy!
    test_loss, acc = test_model(model, test_loader, criterion, test_loss)
    test_acc.append(acc)

print("Final Accuracy: %.2f%%" % acc)

Final Accuracy: 100.00%
